# Feature Selection & Model Fitting

This notebook covers:
1. **VIF analysis** — iterative removal of collinear features
2. **OLS baseline** — linear regression `ROP ~ WOB + RPM`
3. **Log-MLR model** — `log(ROP) ~ WOB + RPM` with residual diagnostics
4. **Bingham physics model** — nonlinear `ROP = K·(WOB/Db)^a·RPM` fitted via MLE


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import scipy.stats as stats
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy.optimize import curve_fit


In [ ]:
# Load dataset (skip the units row)
file_path = "../data/USROP_A 3 N-SH-F-15d - Final.csv"
df = pd.read_csv(file_path, skiprows=[1])
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()


## 1. VIF Analysis — Variable Selection

We start with 7 candidate features and iteratively remove the most collinear
variables until all VIF values are below 5.


In [ ]:
X = df[["WOB", "RPM", "Torque", "TFLO", "HKLD", "SPPA", "MD"]]

vif_data = pd.DataFrame()
vif_data["Variable"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) 
                   for i in range(X.shape[1])]

print(vif_data)

In [ ]:
X1 = df[["WOB", "RPM", "SPPA"]]

vif_1 = pd.DataFrame()
vif_1["Variable"] = X1.columns
vif_1["VIF"] = [variance_inflation_factor(X1.values, i) 
                for i in range(X1.shape[1])]

print(vif_1)


In [ ]:
X2 = df[["WOB", "RPM", "SPPA", "MD"]]

vif_2 = pd.DataFrame()
vif_2["Variable"] = X2.columns
vif_2["VIF"] = [variance_inflation_factor(X2.values, i) 
                for i in range(X2.shape[1])]

print(vif_2)


In [ ]:
X_3 = df[["WOB", "RPM"]]

vif_3 = pd.DataFrame()
vif_3["Variable"] = X_3.columns
vif_3["VIF"] = [variance_inflation_factor(X_3.values, i) 
                    for i in range(X_3.shape[1])]

print(vif_3)


## 2. OLS Baseline Model

Linear regression: `ROP = β₀ + β₁·WOB + β₂·RPM + ε`


In [ ]:
X = df[["WOB", "RPM"]]
X = sm.add_constant(X)   
y = df["ROP"]
model = sm.OLS(y, X).fit()
print(model.summary())

## 3. Log-MLR Model

Log-transformed model: `log(ROP) = β₀ + β₁·WOB + β₂·RPM + ε`

Includes residual analysis and Breusch–Pagan heteroscedasticity test.


In [ ]:
df["log_ROP"] = np.log(df["ROP"])

# Refit model with log(ROP)
y_log = df["log_ROP"]
X = df[["WOB", "RPM"]]
X = sm.add_constant(X)

log_model = sm.OLS(y_log, X).fit()

print(log_model.summary())

In [ ]:
# Get residuals and fitted values
residuals = log_model.resid
fitted = log_model.fittedvalues

plt.figure()
plt.scatter(fitted, residuals, alpha=0.3)
plt.axhline(0)
plt.xlabel("Fitted log(ROP)")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted")
plt.show()

plt.figure()
stats.probplot(residuals, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

bp_test = het_breuschpagan(residuals, log_model.model.exog)

bp_labels = ["LM Stat", "LM p-value", "F-Stat", "F p-value"]
bp_results = dict(zip(bp_labels, bp_test))

print("Breusch–Pagan Test Results:")
print(bp_results)

## 4. Bingham Physics Model (MLE)

The Bingham ROP equation: `ROP = K · (WOB/Db)^a · RPM`

Parameters `K` and `a` are estimated via nonlinear least-squares (MLE).
Bit diameter `Db = 444.5 mm` is constant for this well.


In [ ]:
df_bingham = df[(df["ROP"] > 0) & (df["WOB"] > 0) & (df["RPM"] > 0)].copy()

print("Shape after filtering:", df_bingham.shape)

In [ ]:
def bingham_model(X, K, a):
    wob, rpm = X
    Db = 444.5  # bit diameter in mm (constant)
    return K * (wob / Db)**a * rpm

In [ ]:
# Independent variables
wob = df_bingham["WOB"].values
rpm = df_bingham["RPM"].values

# Target variable
rop_obs = df_bingham["ROP"].values

# Stack predictors for curve_fit
X_data = np.vstack((wob, rpm))


In [ ]:
# Initial guesses (important for nonlinear optimization)
initial_guess = (1.0, 1.0)

params_opt, params_cov = curve_fit(
    bingham_model,
    X_data,
    rop_obs,
    p0=initial_guess,
    maxfev=20000
)
K_hat, a_hat = params_opt

print("Estimated K =", K_hat)
print("Estimated a =", a_hat)